# AI-Powered Student Performance Analytics
## Data Analytics with AI Hackathon
### IBM SkillsBuild | CSRBOX | Edunet Foundation

**Project Goal**: Predict student performance using AI/ML and identify key factors affecting academic success.

---

## Part 1: Data Fundamentals & Exploratory Data Analysis
*(Covers Masterclass 1: Data Fundamentals)*

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Load the dataset
df = pd.read_csv('data/student_performance_raw.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nBasic Statistics:")
print(df.describe())

In [ ]:
# Check for missing values (Data Fundamentals)
print("\n=== MISSING VALUES ANALYSIS ===")
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_data,
    'Percentage': missing_percent
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(missing_df)

# Visualize missing values
plt.figure(figsize=(10, 4))
missing_df['Percentage'].plot(kind='barh', color='coral')
plt.xlabel('Percentage of Missing Values')
plt.title('Missing Data Distribution')
plt.tight_layout()
plt.savefig('outputs/01_missing_values.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Missing values visualization saved")

In [ ]:
# Data type identification (Masterclass 1: Data Fundamentals)
print("\n=== DATA TYPES IDENTIFICATION ===")

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical Columns ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical Columns ({len(categorical_cols)}): {categorical_cols}")

# Identify target variable
print("\n✓ Target Variable: 'Performance_Score' (continuous)")
print("✓ Secondary Target: 'Final_Grade' (categorical)")

## Part 2: Data Cleaning & Preparation
*(Covers Masterclass 2: Data Cleaning & Preparation)*

In [ ]:
# Create a copy for processing
df_clean = df.copy()

print("=== DATA CLEANING PROCESS ===")
print(f"\nOriginal dataset shape: {df_clean.shape}")

# 1. Handle Missing Values
print("\n1. HANDLING MISSING VALUES")

# For numerical columns with missing values: impute with median
for col in numerical_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"   ✓ {col}: filled missing values with median = {median_val:.2f}")

# For categorical columns: impute with mode
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f"   ✓ {col}: filled missing values with mode = {mode_val}")

print(f"\nMissing values after imputation: {df_clean.isnull().sum().sum()}")

In [ ]:
# 2. Detect and Handle Outliers
print("\n2. OUTLIER DETECTION & HANDLING")

outlier_cols = ['Study_Hours_Per_Day', 'Sleep_Hours', 'Distance_From_College_KM']

for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
    
    # Cap outliers instead of removing
    df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)
    
    print(f"   ✓ {col}: {outliers} outliers detected and capped")
    print(f"      Range: [{lower_bound:.2f}, {upper_bound:.2f}]")

In [ ]:
# 3. Data Validation & Type Conversion
print("\n3. DATA VALIDATION & TYPE CONVERSION")

# Ensure Age is in valid range
df_clean = df_clean[(df_clean['Age'] >= 17) & (df_clean['Age'] <= 25)]
print(f"   ✓ Age validation: removed {len(df) - len(df_clean)} invalid records")

# Ensure percentages are 0-100
df_clean['Attendance_Percentage'] = df_clean['Attendance_Percentage'].clip(0, 100)
df_clean['Assignment_Completion_Rate'] = df_clean['Assignment_Completion_Rate'].clip(0, 100)
print("   ✓ Percentage columns validated (0-100 range)")

# Ensure Performance_Score is 0-100
df_clean['Performance_Score'] = df_clean['Performance_Score'].clip(0, 100)
print("   ✓ Performance_Score validated (0-100 range)")

print(f"\nCleaned dataset shape: {df_clean.shape}")
print("\n✓ Data Cleaning Complete!")

In [ ]:
# 4. Exploratory Data Analysis (EDA)
print("\n4. EXPLORATORY DATA ANALYSIS (EDA)")

# Numerical summary statistics
print("\nNumerical Features Summary:")
print(df_clean[numerical_cols].describe().round(2))

# Categorical summary
print("\n\nCategorical Features Distribution:")
for col in categorical_cols:
    print(f"\n{col}:")
    print(df_clean[col].value_counts())

In [ ]:
# Distribution analysis
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
key_numerical = ['Attendance_Percentage', 'Study_Hours_Per_Day', 'Previous_CGPA', 
                 'Assignment_Completion_Rate', 'Mental_Health_Score', 'Performance_Score',
                 'Age', 'Sleep_Hours', 'Distance_From_College_KM']

for idx, col in enumerate(key_numerical):
    ax = axes[idx // 3, idx % 3]
    ax.hist(df_clean[col], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('outputs/02_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Distribution visualizations saved")

## Part 3: Feature Engineering & Analysis
*(Prepares for AI/ML - Masterclass 3)*

In [ ]:
# Correlation analysis with target variable
print("\n=== CORRELATION ANALYSIS ===")

correlations = df_clean[numerical_cols].corr()['Performance_Score'].sort_values(ascending=False)
print("\nCorrelation with Performance_Score:")
print(correlations)

# Visualize correlations
plt.figure(figsize=(10, 8))
correlation_matrix = df_clean[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.savefig('outputs/03_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Correlation matrix visualization saved")

In [ ]:
# Categorical analysis
print("\n=== CATEGORICAL ANALYSIS ===")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Department analysis
dept_performance = df_clean.groupby('Department')['Performance_Score'].agg(['mean', 'std', 'count'])
axes[0, 0].bar(dept_performance.index, dept_performance['mean'], color='steelblue', alpha=0.7)
axes[0, 0].set_title('Average Performance by Department')
axes[0, 0].set_ylabel('Avg Performance Score')
axes[0, 0].tick_params(axis='x', rotation=45)

# Gender analysis
gender_performance = df_clean.groupby('Gender')['Performance_Score'].mean()
axes[0, 1].bar(gender_performance.index, gender_performance.values, color=['#FF69B4', '#4169E1', '#FFD700'], alpha=0.7)
axes[0, 1].set_title('Average Performance by Gender')
axes[0, 1].set_ylabel('Avg Performance Score')

# Year of Study analysis
year_performance = df_clean.groupby('Year_of_Study')['Performance_Score'].mean()
axes[1, 0].plot(year_performance.index, year_performance.values, marker='o', linewidth=2, markersize=8, color='green')
axes[1, 0].set_title('Performance Trend by Year of Study')
axes[1, 0].set_xlabel('Year of Study')
axes[1, 0].set_ylabel('Avg Performance Score')
axes[1, 0].grid(True, alpha=0.3)

# Internet Access impact
internet_performance = df_clean.groupby('Internet_Access')['Performance_Score'].mean()
axes[1, 1].bar(internet_performance.index, internet_performance.values, color=['#FF6347', '#3CB371'], alpha=0.7)
axes[1, 1].set_title('Performance Impact: Internet Access')
axes[1, 1].set_ylabel('Avg Performance Score')

plt.tight_layout()
plt.savefig('outputs/04_categorical_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Categorical analysis visualizations saved")

In [ ]:
# Key insights from EDA
print("\n" + "="*60)
print("KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("="*60)

print(f"\n1. Top 3 Factors Affecting Performance:")
for i, (col, corr) in enumerate(correlations.head(4).items(), 1):
    if col != 'Performance_Score':
        print(f"   {i}. {col}: {corr:.3f}")

best_dept = dept_performance['mean'].idxmax()
worst_dept = dept_performance['mean'].idxmin()
print(f"\n2. Department Performance:")
print(f"   Best: {best_dept} (avg: {dept_performance.loc[best_dept, 'mean']:.2f})")
print(f"   Needs Improvement: {worst_dept} (avg: {dept_performance.loc[worst_dept, 'mean']:.2f})")

avg_attendance = df_clean['Attendance_Percentage'].mean()
print(f"\n3. Average Attendance: {avg_attendance:.2f}%")

internet_impact = internet_performance['Yes'] - internet_performance['No']
print(f"\n4. Internet Access Impact: +{internet_impact:.2f} points")

high_performers = (df_clean['Performance_Score'] >= 80).sum()
print(f"\n5. High Performers (Score ≥ 80): {high_performers} students ({high_performers/len(df_clean)*100:.1f}%)")

## Part 4: AI/ML Model Building
*(Covers Masterclass 3: AI for Data Analytics)*

In [ ]:
# Prepare data for modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb

print("\n=== PREPARING DATA FOR ML MODELS ===")

# Encode categorical variables
df_model = df_clean.copy()
le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    le_dict[col] = le
    print(f"✓ Encoded {col}: {len(le.classes_)} categories")

# Separate features and target
X = df_model.drop(['Performance_Score', 'Final_Grade'], axis=1)
y = df_model['Performance_Score']

print(f"\n✓ Features shape: {X.shape}")
print(f"✓ Target shape: {y.shape}")
print(f"\nFeatures: {X.columns.tolist()}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n=== TRAIN-TEST SPLIT ===")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"Test size: {len(X_test) / len(X) * 100:.1f}%")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Features scaled using StandardScaler")

In [ ]:
# Train multiple models
print("\n=== TRAINING ML MODELS ===")

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
}

model_results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    model_results[name] = {
        'model': model,
        'predictions': y_pred,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }
    
    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R² Score: {r2:.4f}")

In [ ]:
# Model comparison
results_df = pd.DataFrame({
    'Model': list(model_results.keys()),
    'RMSE': [model_results[m]['RMSE'] for m in model_results.keys()],
    'MAE': [model_results[m]['MAE'] for m in model_results.keys()],
    'R2': [model_results[m]['R2'] for m in model_results.keys()]
})

print("\n=== MODEL COMPARISON ===")
print(results_df.to_string(index=False))

best_model_name = results_df.loc[results_df['R2'].idxmax(), 'Model']
print(f"\n🏆 BEST MODEL: {best_model_name}")

# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, metric in enumerate(['RMSE', 'MAE', 'R2']):
    axes[idx].bar(results_df['Model'], results_df[metric], color='steelblue', alpha=0.7)
    axes[idx].set_title(f'Model {metric} Comparison')
    axes[idx].set_ylabel(metric)
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('outputs/05_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Model comparison visualization saved")

In [ ]:
# Feature importance from best model (Random Forest or XGBoost)
best_model = model_results[best_model_name]['model']

if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print(f"\n=== FEATURE IMPORTANCE ({best_model_name}) ===")
    print(feature_importance.head(10).to_string(index=False))
    
    # Visualize feature importance
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['Feature'][:10], feature_importance['Importance'][:10], color='steelblue')
    plt.xlabel('Importance Score')
    plt.title(f'Top 10 Most Important Features ({best_model_name})')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('outputs/06_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("\n✓ Feature importance visualization saved")

In [ ]:
# Actual vs Predicted
best_predictions = model_results[best_model_name]['predictions']

plt.figure(figsize=(10, 6))
plt.scatter(y_test, best_predictions, alpha=0.5, s=30)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Performance Score')
plt.ylabel('Predicted Performance Score')
plt.title(f'Actual vs Predicted Performance ({best_model_name})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/07_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Actual vs Predicted visualization saved")

## Part 5: Business Intelligence & Dashboard Insights
*(Covers Masterclass 4: Business Solutions with IBM Cognos style)*

In [ ]:
# Segment students into risk categories based on predictions
print("\n=== AT-RISK STUDENT IDENTIFICATION ===")

# Create predictions on full dataset
X_full_scaled = scaler.transform(X)
full_predictions = best_model.predict(X_full_scaled)

df_clean['Predicted_Score'] = full_predictions

# Categorize risk levels
def risk_category(score):
    if score >= 80:
        return 'High Performer'
    elif score >= 60:
        return 'Average'
    else:
        return 'At-Risk'

df_clean['Risk_Category'] = df_clean['Predicted_Score'].apply(risk_category)

# Risk distribution
risk_dist = df_clean['Risk_Category'].value_counts()
print("\nStudent Risk Distribution:")
print(risk_dist)
print(f"\nAt-Risk Students: {risk_dist.get('At-Risk', 0)} ({risk_dist.get('At-Risk', 0)/len(df_clean)*100:.1f}%)")
print(f"High Performers: {risk_dist.get('High Performer', 0)} ({risk_dist.get('High Performer', 0)/len(df_clean)*100:.1f}%)")

In [ ]:
# Comprehensive dashboard visualization
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Risk Category Distribution
ax1 = fig.add_subplot(gs[0, 0])
colors_risk = {'At-Risk': '#FF6B6B', 'Average': '#FFD93D', 'High Performer': '#6BCB77'}
risk_dist.plot(kind='bar', ax=ax1, color=[colors_risk.get(cat, 'gray') for cat in risk_dist.index])
ax1.set_title('Student Risk Distribution', fontweight='bold')
ax1.set_ylabel('Number of Students')
ax1.tick_params(axis='x', rotation=45)

# 2. Department Performance
ax2 = fig.add_subplot(gs[0, 1])
dept_risk = pd.crosstab(df_clean['Department'], df_clean['Risk_Category'])
dept_risk.plot(kind='bar', ax=ax2, color=[colors_risk.get(cat, 'gray') for cat in dept_risk.columns])
ax2.set_title('Risk Distribution by Department', fontweight='bold')
ax2.set_ylabel('Number of Students')
ax2.tick_params(axis='x', rotation=45)
ax2.legend(title='Risk')

# 3. Gender Analysis
ax3 = fig.add_subplot(gs[0, 2])
gender_perf = df_clean.groupby('Gender')['Predicted_Score'].mean()
ax3.bar(gender_perf.index, gender_perf.values, color=['#FF69B4', '#4169E1', '#FFD700'])
ax3.set_title('Average Predicted Performance by Gender', fontweight='bold')
ax3.set_ylabel('Avg Score')

# 4. Attendance Impact
ax4 = fig.add_subplot(gs[1, 0])
attendance_bins = [0, 60, 75, 90, 100]
df_clean['Attendance_Group'] = pd.cut(df_clean['Attendance_Percentage'], bins=attendance_bins)
att_perf = df_clean.groupby('Attendance_Group')['Predicted_Score'].mean()
ax4.plot(range(len(att_perf)), att_perf.values, marker='o', linewidth=2, markersize=8)
ax4.set_title('Performance vs Attendance Level', fontweight='bold')
ax4.set_ylabel('Avg Predicted Score')
ax4.set_xticklabels(['0-60%', '60-75%', '75-90%', '90-100%'])
ax4.grid(True, alpha=0.3)

# 5. Study Hours Impact
ax5 = fig.add_subplot(gs[1, 1])
study_bins = [0, 2, 4, 6, 10]
df_clean['Study_Group'] = pd.cut(df_clean['Study_Hours_Per_Day'], bins=study_bins)
study_perf = df_clean.groupby('Study_Group')['Predicted_Score'].mean()
ax5.bar(range(len(study_perf)), study_perf.values, color='steelblue')
ax5.set_title('Performance vs Daily Study Hours', fontweight='bold')
ax5.set_ylabel('Avg Predicted Score')
ax5.set_xticklabels(['0-2h', '2-4h', '4-6h', '6-10h'])

# 6. Internet Access Impact
ax6 = fig.add_subplot(gs[1, 2])
internet_comp = df_clean.groupby('Internet_Access')['Predicted_Score'].mean()
ax6.bar(internet_comp.index, internet_comp.values, color=['#FF6347', '#3CB371'])
ax6.set_title('Internet Access Impact', fontweight='bold')
ax6.set_ylabel('Avg Predicted Score')

# 7. Mental Health Impact
ax7 = fig.add_subplot(gs[2, 0])
mental_health_avg = df_clean.groupby('Mental_Health_Score')['Predicted_Score'].mean()
ax7.plot(mental_health_avg.index, mental_health_avg.values, marker='o', linewidth=2)
ax7.set_title('Mental Health Score Impact', fontweight='bold')
ax7.set_xlabel('Mental Health Score')
ax7.set_ylabel('Avg Predicted Score')
ax7.grid(True, alpha=0.3)

# 8. Extracurricular Activities
ax8 = fig.add_subplot(gs[2, 1])
extra_curr = df_clean.groupby('Extracurricular_Activities')['Predicted_Score'].mean()
ax8.bar(extra_curr.index, extra_curr.values, color='coral')
ax8.set_title('Impact of Extracurricular Activities', fontweight='bold')
ax8.set_xlabel('Number of Activities')
ax8.set_ylabel('Avg Predicted Score')

# 9. Year-wise Performance
ax9 = fig.add_subplot(gs[2, 2])
year_perf = df_clean.groupby('Year_of_Study')['Predicted_Score'].mean()
ax9.plot(year_perf.index, year_perf.values, marker='s', linewidth=2, markersize=8, color='green')
ax9.set_title('Performance Trend: Year of Study', fontweight='bold')
ax9.set_xlabel('Year of Study')
ax9.set_ylabel('Avg Predicted Score')
ax9.grid(True, alpha=0.3)

plt.suptitle('AI-Powered Student Performance Analytics Dashboard', fontsize=16, fontweight='bold', y=0.995)
plt.savefig('outputs/08_comprehensive_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Comprehensive dashboard visualization saved")

## Part 6: Business Recommendations & Conclusions

In [ ]:
print("\n" + "="*70)
print("BUSINESS INTELLIGENCE & ACTIONABLE INSIGHTS")
print("="*70)

# At-risk students analysis
at_risk = df_clean[df_clean['Risk_Category'] == 'At-Risk']
high_perf = df_clean[df_clean['Risk_Category'] == 'High Performer']

print("\n1. AT-RISK STUDENT PROFILE:")
print(f"   - Count: {len(at_risk)} students ({len(at_risk)/len(df_clean)*100:.1f}%)")
print(f"   - Avg Attendance: {at_risk['Attendance_Percentage'].mean():.1f}%")
print(f"   - Avg Study Hours/Day: {at_risk['Study_Hours_Per_Day'].mean():.1f} hours")
print(f"   - Avg Previous CGPA: {at_risk['Previous_CGPA'].mean():.2f}")
print(f"   - Mental Health Score: {at_risk['Mental_Health_Score'].mean():.1f}/10")
print(f"   - Internet Access: {(at_risk['Internet_Access']=='Yes').sum() / len(at_risk) * 100:.1f}% have access")

print("\n2. HIGH PERFORMER PROFILE:")
print(f"   - Count: {len(high_perf)} students ({len(high_perf)/len(df_clean)*100:.1f}%)")
print(f"   - Avg Attendance: {high_perf['Attendance_Percentage'].mean():.1f}%")
print(f"   - Avg Study Hours/Day: {high_perf['Study_Hours_Per_Day'].mean():.1f} hours")
print(f"   - Avg Previous CGPA: {high_perf['Previous_CGPA'].mean():.2f}")
print(f"   - Mental Health Score: {high_perf['Mental_Health_Score'].mean():.1f}/10")
print(f"   - Internet Access: {(high_perf['Internet_Access']=='Yes').sum() / len(high_perf) * 100:.1f}% have access")

print("\n3. KEY PERFORMANCE DRIVERS (Feature Importance):")
if 'feature_importance' in locals():
    for i, row in feature_importance.head(5).iterrows():
        print(f"   {i+1}. {row['Feature']}: {row['Importance']:.4f}")

print("\n4. DEPARTMENTAL INSIGHTS:")
dept_stats = df_clean.groupby('Department').agg({
    'Predicted_Score': ['mean', 'std'],
    'Student_ID': 'count'
}).round(2)
dept_stats.columns = ['Avg_Score', 'Std_Dev', 'Count']
dept_stats = dept_stats.sort_values('Avg_Score', ascending=False)
print(dept_stats.to_string())

In [ ]:
print("\n" + "="*70)
print("STRATEGIC RECOMMENDATIONS")
print("="*70)

recommendations = [
    {
        'title': '1. EARLY WARNING SYSTEM',
        'details': [
            f"  • Implement automated alerts for students with predicted scores < 60",
            f"  • Current at-risk count: {len(at_risk)} students",
            f"  • Trigger interventions within first month of semester"
        ]
    },
    {
        'title': '2. ATTENDANCE OPTIMIZATION',
        'details': [
            f"  • Focus on maintaining attendance above 75%",
            f"  • At-risk students avg attendance: {at_risk['Attendance_Percentage'].mean():.1f}%",
            f"  • High performers avg attendance: {high_perf['Attendance_Percentage'].mean():.1f}%",
            f"  • Performance gain: +{high_perf['Attendance_Percentage'].mean() - at_risk['Attendance_Percentage'].mean():.1f}% attendance = +{high_perf['Predicted_Score'].mean() - at_risk['Predicted_Score'].mean():.1f} score"
        ]
    },
    {
        'title': '3. STUDY HOUR TARGETS',
        'details': [
            f"  • Recommended daily study hours: 4-6 hours",
            f"  • At-risk students avg: {at_risk['Study_Hours_Per_Day'].mean():.1f} hours/day",
            f"  • High performers avg: {high_perf['Study_Hours_Per_Day'].mean():.1f} hours/day",
            f"  • Provide study skill workshops and time management training"
        ]
    },
    {
        'title': '4. MENTAL HEALTH SUPPORT',
        'details': [
            f"  • At-risk students mental health score: {at_risk['Mental_Health_Score'].mean():.1f}/10",
            f"  • High performers mental health score: {high_perf['Mental_Health_Score'].mean():.1f}/10",
            f"  • Strengthen counseling and wellness programs",
            f"  • Create peer support groups"
        ]
    },
    {
        'title': '5. DIGITAL INFRASTRUCTURE',
        'details': [
            f"  • {100 - (df_clean['Internet_Access']=='Yes').sum()/len(df_clean)*100:.1f}% of students lack reliable internet",
            f"  • Provide free WiFi zones and digital resources",
            f"  • Offer offline-friendly learning materials"
        ]
    },
    {
        'title': '6. DEPARTMENT-SPECIFIC INITIATIVES',
        'details': [
            f"  • Lowest performing dept: {dept_stats.index[-1]} (avg: {dept_stats.iloc[-1, 0]:.1f})",
            f"  • Allocate extra resources and mentorship",
            f"  • Best performing dept: {dept_stats.index[0]} - use as model"
        ]
    }
]

for rec in recommendations:
    print(f"\n{rec['title']}")
    for detail in rec['details']:
        print(detail)

In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY & CONCLUSIONS")
print("="*70)

print(f"""
✓ DATASET ANALYSIS:
  • Total Students Analyzed: {len(df_clean)}
  • Features: {len(X.columns)}
  • Missing Values: {df_clean.isnull().sum().sum()} (after imputation)
  • Outliers Detected & Handled: Yes

✓ DATA QUALITY:
  • Data Cleaning: Comprehensive (missing values, outliers, validation)
  • Feature Engineering: Feature importance identified
  • Statistical Analysis: Correlations, distributions, segmentation

✓ AI/ML MODELS TRAINED:
  • Models: {', '.join(list(models.keys()))}
  • Best Model: {best_model_name}
  • R² Score: {model_results[best_model_name]['R2']:.4f}
  • RMSE: {model_results[best_model_name]['RMSE']:.4f}
  • MAE: {model_results[best_model_name]['MAE']:.4f}

✓ BUSINESS INSIGHTS:
  • At-Risk Students: {len(at_risk)} ({len(at_risk)/len(df_clean)*100:.1f}%)
  • High Performers: {len(high_perf)} ({len(high_perf)/len(df_clean)*100:.1f}%)
  • Key Success Factor: Attendance & Study Hours
  • Mental Health Score Impact: Significant positive correlation

✓ DELIVERABLES:
  • 8 visualization outputs (PNG files)
  • Predictive model trained and validated
  • Student risk segmentation completed
  • Actionable business recommendations
  • Ready for implementation in educational institutions

🎯 NEXT STEPS FOR INSTITUTIONS:
  1. Deploy early warning system using predictions
  2. Implement targeted interventions
  3. Monitor and track student progress
  4. Refine model with new data quarterly
  5. Expand to other institutions/departments
""")

print("="*70)
print("PROJECT COMPLETION: ✓ SUCCESS")
print("="*70)

In [ ]:
# Save processed data for reference
df_clean.to_csv('data/student_performance_cleaned.csv', index=False)
print("✓ Cleaned dataset saved: data/student_performance_cleaned.csv")

# Save model results summary
results_df.to_csv('outputs/model_results_summary.csv', index=False)
print("✓ Model results summary saved: outputs/model_results_summary.csv")

print("\n✓ All outputs generated successfully!")
print("\nOutput Directory Structure:")
print("  outputs/")
print("    01_missing_values.png")
print("    02_distributions.png")
print("    03_correlation_matrix.png")
print("    04_categorical_analysis.png")
print("    05_model_comparison.png")
print("    06_feature_importance.png")
print("    07_actual_vs_predicted.png")
print("    08_comprehensive_dashboard.png")
print("    model_results_summary.csv")